# USIA — Interactive pipeline explorerThe pipeline lives in `analysis/src/*.py`. Those files stay the source of truth; thisnotebook drives them and explores the outputs, so nothing is duplicated (seeCODING_STANDARDS: *share via `src/`, don't copy-paste across notebooks*).Script filenames start with digits, so they can't be `import`ed as modules. Use`%run` to execute one end to end, or `load()` below to reach individual functions.**Run order**1. `01_build_intervention_dates.py` — works now, no downloads needed2. `03_bikelane_parking_overlap.py` — kerb capacity works now; overlap needs the BIN file3. `04_figures.py` — works now4. `00_download_com_data.py` → `02_sensor_utilisation.py` — needs the CoM archives

## 0. Setup

In [ ]:
%load_ext autoreload%autoreload 2import sys, importlib.utilfrom pathlib import Path# analysis/ must be importable so `import config` works the same way the scripts doANALYSIS = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ANALYSIS))import config as Cimport pandas as pd, numpy as np, geopandas as gpdimport matplotlib.pyplot as pltpd.set_option("display.width", 200)pd.set_option("display.max_columns", 50)print("analysis dir :", ANALYSIS)print("project dir  :", C.PROJ)print("geopandas    :", gpd.__version__)print("streets gpkg :", C.STREETS_GPKG.exists())

In [ ]:
def load(script_name):    """Import a numerically-prefixed pipeline script as a module.    Parameters:        script_name (str): filename in analysis/src, e.g. '01_build_intervention_dates.py'    Returns:        module: the loaded module, so you can call its functions directly.    """    path = ANALYSIS / "src" / script_name    spec = importlib.util.spec_from_file_location(path.stem, path)    mod = importlib.util.module_from_spec(spec)    spec.loader.exec_module(mod)    return mod

## 1. Intervention datesDecodes IV's financial-year quarter codes and flags which segments the sensor archivecan actually support. Runs on data already in the repo.

In [ ]:
%run "$ANALYSIS/src/01_build_intervention_dates.py"

In [ ]:
dates = pd.read_csv(C.PROCESSED / "intervention_dates.csv", dtype={"sid": str},                    parse_dates=["intervention_date"])print("sensor-usable segments:", int(dates.sensor_usable.sum()))dates.loc[dates.sensor_usable, ["street_name", "sid", "intervention_date",                                "sensor_pre_days", "sensor_post_days"]]

### Segments with no intervention date — the query list for IV

In [ ]:
dates.loc[dates.missing_intervention_date,          ["sid", "street_name", "suburb", "lga", "intervention_type"]]

## 2. Kerb capacity and the Grattan obstruction factorCalibrates the Appendix D kerb-length-to-spaces conversion against IV's own consultantcounts. The `--bin` argument is optional; without it the overlap step is skipped.

In [ ]:
%run "$ANALYSIS/src/03_bikelane_parking_overlap.py"

In [ ]:
cap = pd.read_csv(C.OUTPUTS / "segment_kerb_capacity.csv")cap["loc_type"] = np.select([cap.cbd == 1, cap.metro == 1, cap.regional == 1],                            ["CBD", "Metro", "Regional"], "?")cap["factor"] = cap.cap_reported / cap.cap_geometricprint(f"total kerb      : {cap.kerb_length_m.sum()/1000:,.1f} km")print(f"total capacity  : {cap.cap_best.sum():,.0f} spaces")cap.groupby("loc_type").factor.describe()[["count", "25%", "50%", "75%"]].round(3)

The CBD/Metro gap (≈0.25 vs 0.44) is why a single global factor overstates CBDcapacity. Fit it per location type before this goes to IV.

## 3. FiguresWrites three diagnostic PNGs to `outputs/figures/`.

In [ ]:
%run "$ANALYSIS/src/04_figures.py"

In [ ]:
from IPython.display import Image, displayfor p in sorted((C.OUTPUTS / "figures").glob("*.png")):    print(p.name)    display(Image(str(p)))

## 4. Sensor data — requires downloads~9.8 GB across ten archives, so run these from a terminal rather than a notebook cell;you want to be able to interrupt them. Uncomment to run here.```bashpython src/00_download_com_data.py --referencepython src/00_download_com_data.py --years 2016 2017 2018 2019   # ~3.9 GB, William Stpython src/02_sensor_utilisation.py --years 2016 2017 2018 2019```

In [ ]:
# Inspect the archive inventory without downloading anythingdl = load("00_download_com_data.py")inv = pd.DataFrame(    [(y, i, s, r) for y, (i, s, r) in dl.ARCHIVES.items()],    columns=["year", "socrata_id", "size", "rows"],)print(f"total rows: {inv.rows.sum()/1e6:,.1f} M")inv

## 5. ScratchWork below here. Anything reusable should move into `src/` rather than living in thenotebook — that's the DRY rule in CODING_STANDARDS.

In [ ]:
# scratch